In [ ]:
#| hide
from mcpmini.core import *

# mcpmini

> A small readable MCP library for serving Python functions as tools and calling MCP servers

mcpmini is a small, readable MCP server and client library. It exposes Python functions as tools over stdio or streamable HTTP, deriving their schemas from signatures and parameter documentation. Its client exposes remote tools as Python callables with reconstructed signatures, documentation, and defaults.


## Usage

### Installation

```sh
pip install mcpmini
```

## How to use

Define a Python function with type annotations and docments, the comments beside parameters. mcpmini uses these to build the tool schema. Pass the functions to `MCPServer`:


In [ ]:
from mcpmini.core import MCPServer, MCPClient, serve_stdio, serve_mcp
import asyncio, socket

In [ ]:
def fahrenheit(
    celsius:float, # Temperature to convert
)->float:
    "Convert Celsius to Fahrenheit"
    return celsius*9/5+32

srv = MCPServer('demo', [fahrenheit])


Choose a transport for the server:

- `asyncio.run(serve_stdio(srv))` serves it over stdio for an MCP host to launch.
- `asyncio.run(serve_mcp(srv, port=8000, token='S'))` serves it over streamable HTTP with bearer-token authentication. A non-loopback bind requires a token unless `no_token=True` explicitly disables that requirement.

On stdio, a tool can call `await srv.elicit(message, schema)` to request structured input while its tool call remains active. A client created with `MCPClient.stdio(..., on_request=handler)` supplies the answer.

The CLI serves functions from a file:

```sh
mcpmini tools.py
mcpmini tools.py --transport http --port 8000
```

Use these commands in an MCP host's server configuration. To connect Claude Code to the HTTP server configured with token `S` above:

```sh
claude mcp add --transport http demo http://127.0.0.1:8000/mcp -H "Authorization: Bearer S"
```

`MCPClient` exposes server tools as bound Python functions. The following example starts a local server and calls its `fahrenheit` tool:

In [ ]:
def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]
port = free_port()
task = asyncio.create_task(serve_mcp(srv, port=port, token='S'))
await asyncio.sleep(0.2)
async with MCPClient.http(f'http://127.0.0.1:{port}/mcp', token='S') as c: res = await c.tools.fahrenheit(celsius=100)
task.cancel()
res

'212.0'